# Apêndice A - Introdução ao PyTorch

Notebook do Apêndice A do livro *Build a Large Language Model (From
Scratch)*, de Sebastian Raschka, seguindo a ordem dos notebooks do autor
(`appendix-A/01_main-chapter-code/code-part1.ipynb` e `code-part2.ipynb`).

Diferença em relação ao livro: a rede, o dataset de brinquedo e o laço de
treino ficam em `src/pytorch_basico/` e são **importados** aqui. A escolha
do dispositivo reaproveita `src/training/dispositivo.py` (Sprint 5).

In [ ]:
import sys
from pathlib import Path

# O notebook roda a partir de Apendices/appendix-A/notebooks/: a raiz do
# projeto fica tres niveis acima.
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from importlib.metadata import version

import numpy as np
import torch
import torch.nn.functional as F

print("torch version:", version("torch"))

## A.1 O que é o PyTorch

- Uma **biblioteca de tensores** (como o NumPy, mas com GPU).
- Um motor de **diferenciação automática** (autograd): calcula gradientes
  sozinho a partir das operações feitas.
- Uma biblioteca de **deep learning** (`torch.nn`, otimizadores, DataLoader).

In [ ]:
from src.training.dispositivo import descrever_dispositivo, escolher_dispositivo

device = escolher_dispositivo()
print("GPU CUDA disponivel:", torch.cuda.is_available())
print("Dispositivo escolhido:", descrever_dispositivo(device))

## A.2 Tensores

### A.2.1 Escalares, vetores, matrizes e tensores

Um tensor generaliza escalar (0D), vetor (1D) e matriz (2D) para qualquer
número de dimensões. `torch.tensor` **copia** um array NumPy;
`torch.from_numpy` **compartilha** a mesma memória.

In [ ]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2], [3, 4]])
tensor3d_1 = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

ary3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
tensor3d_2 = torch.tensor(ary3d)      # copia
tensor3d_3 = torch.from_numpy(ary3d)  # compartilha memoria

ary3d[0, 0, 0] = 999
print("torch.tensor    :", tensor3d_2[0, 0, 0].item(), "(nao muda)")
print("torch.from_numpy:", tensor3d_3[0, 0, 0].item(), "(muda junto)")

### A.2.2 Tipos de dados

Inteiros viram `int64`; números de ponto flutuante viram `float32`, que tem
precisão suficiente para deep learning e é mais rápido na GPU que `float64`.

In [ ]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)
print(torch.tensor([1.0, 2.0, 3.0]).dtype)
print(tensor1d.to(torch.float32).dtype)

### A.2.3 Operações comuns

`reshape` e `view` mudam o formato; `view` exige memória contígua, e a
transposta `.T` não é contígua. `@` é a multiplicação de matrizes, a
operação mais usada na atenção (cap. 3).

In [ ]:
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("shape      :", tensor2d.shape)
print("reshape    :", tensor2d.reshape(3, 2).tolist())
print("view       :", tensor2d.view(3, 2).tolist())
print("T          :", tensor2d.T.tolist())
print("matmul     :", tensor2d.matmul(tensor2d.T).tolist())
print("@          :", (tensor2d @ tensor2d.T).tolist())
print("T contigua?:", tensor2d.T.is_contiguous())

## A.3 Modelos como grafos computacionais

Uma regressão logística de um neurônio: `z = x1·w1 + b`, `a = sigmoid(z)`
e a perda é a entropia cruzada binária entre `a` e o rótulo `y`. O PyTorch
monta o **grafo** dessas operações enquanto elas acontecem.

In [ ]:
y = torch.tensor([1.0])   # rotulo verdadeiro
x1 = torch.tensor([1.1])  # feature de entrada
w1 = torch.tensor([2.2])  # peso
b = torch.tensor([0.0])   # vies

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
print(loss)

## A.4 Diferenciação automática

Com `requires_grad=True`, o PyTorch guarda o grafo e calcula os
gradientes pela regra da cadeia. Há dois jeitos: `torch.autograd.grad`
(explícito) e `loss.backward()` (preenche o `.grad` de cada folha, que é o
que os laços de treino usam).

Para sigmoid com entropia cruzada binária, a derivada em relação a `z` é
simplesmente `a − y`, então `dL/dw1 = (a − y)·x1` e `dL/db = a − y`.

In [ ]:
from torch.autograd import grad

w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
a = torch.sigmoid(x1 * w1 + b)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)
print("autograd.grad:", grad_L_w1, grad_L_b)

loss.backward()
print("backward()   :", w1.grad, b.grad)
print("a mao        :", ((a - y) * x1).item(), (a - y).item())

## A.5 Redes neurais multicamadas

`NeuralNetwork` (em `src/pytorch_basico/rede_neural.py`) tem duas camadas
ocultas com 30 e 20 neurônios e ReLU. A saída são **logits**: a softmax
fica de fora porque `F.cross_entropy` já a aplica internamente.

In [ ]:
from src.pytorch_basico.rede_neural import (
    NeuralNetwork,
    contar_parametros_treinaveis,
    parametros_esperados,
)

model = NeuralNetwork(50, 3)
print(model)
print("Parametros treinaveis:", contar_parametros_treinaveis(model))
print("Conta a mao          :", parametros_esperados(50, 3))

In [ ]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print("Shape dos pesos da 1a camada:", model.layers[0].weight.shape)

torch.manual_seed(123)
X = torch.rand((1, 50))
print("Com grafo   :", model(X))            # grad_fn=<AddmmBackward0>
with torch.no_grad():
    print("Sem grafo   :", model(X))
    print("Softmax     :", torch.softmax(model(X), dim=1))

## A.6 DataLoaders eficientes

Um `Dataset` só precisa de `__getitem__` e `__len__`; o `DataLoader` monta
os lotes, embaralha (`shuffle`) e pode descartar o último lote incompleto
(`drop_last`). É o mesmo esquema do `GPTDatasetV1` do cap. 2.

In [ ]:
from src.pytorch_basico.dados_toy import ToyDataset, criar_loaders_toy, dados_toy

X_train, y_train, X_test, y_test = dados_toy()
train_ds = ToyDataset(X_train, y_train)
print("len(train_ds):", len(train_ds))

for drop_last in (False, True):
    train_loader, _ = criar_loaders_toy(batch_size=2, drop_last=drop_last)
    print(f"\ndrop_last={drop_last}")
    for idx, (x, y_lote) in enumerate(train_loader):
        print(f"  Batch {idx + 1}:", x.tolist(), y_lote.tolist())

## A.7 Um laço de treino típico

Para cada lote: forward, perda, `zero_grad`, `backward`, `step`. Sem o
`zero_grad`, os gradientes de lotes diferentes se **somariam**.

In [ ]:
from src.pytorch_basico.treino_basico import compute_accuracy, treinar_rede

train_loader, test_loader = criar_loaders_toy()
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

perdas = treinar_rede(model, train_loader, optimizer, num_epochs=3)

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)

torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
predictions = torch.argmax(probas, dim=1)
print(probas)
print("Previsoes:", predictions.tolist(), " Rotulos:", y_train.tolist())
print("Acuracia treino:", compute_accuracy(model, train_loader))
print("Acuracia teste :", compute_accuracy(model, test_loader))

## A.8 Salvar e carregar modelos

O `state_dict` é um dicionário `nome da camada → tensor`. Para carregar, a
arquitetura precisa ser **idêntica**. O arquivo vai para `models/`, que
está fora do git.

In [ ]:
caminho = PROJECT_ROOT / "models" / "appendix-A" / "model.pth"
caminho.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), caminho)

novo = NeuralNetwork(2, 2)  # precisa ter a mesma arquitetura
novo.load_state_dict(torch.load(caminho, weights_only=True))
novo.eval()
with torch.no_grad():
    print("Saidas identicas:", torch.equal(model(X_train), novo(X_train)))
print("Chaves:", list(novo.state_dict()))

## A.9 Treinando com GPU

### A.9.1 e A.9.2 Uma GPU

Só três mudanças em relação ao laço da A.7: escolher o `device`, mover o
modelo com `model.to(device)` e mover cada lote com `.to(device)`. Tensores
em dispositivos diferentes não podem ser combinados.

In [ ]:
torch.manual_seed(123)
model = NeuralNetwork(2, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
treinar_rede(model, train_loader, optimizer, device=device, num_epochs=3, verbose=False)

print("Pesos em:", next(model.parameters()).device)
print("Acuracia teste:", compute_accuracy(model, test_loader, device=device))

In [ ]:
import time

def tempo_matmul(dev, n=2048, repeticoes=5):
    a, b = torch.rand(n, n, device=dev), torch.rand(n, n, device=dev)
    a @ b
    if dev.type == "cuda":
        torch.cuda.synchronize()
    tempos = []
    for _ in range(repeticoes):
        t0 = time.perf_counter()
        a @ b
        if dev.type == "cuda":
            torch.cuda.synchronize()  # a GPU roda assincrona
        tempos.append(time.perf_counter() - t0)
    return min(tempos) * 1000

print(f"CPU: {tempo_matmul(torch.device('cpu')):.1f} ms")
if device.type != "cpu":
    print(f"{device.type.upper()}: {tempo_matmul(device):.1f} ms")

### A.9.3 Várias GPUs (DDP)

O *DistributedDataParallel* copia o modelo em cada GPU, divide os lotes
entre elas (`DistributedSampler`) e sincroniza os gradientes. Não é
reproduzido aqui: exige duas ou mais GPUs e o backend NCCL, que não existe
no Windows. Os scripts do autor ficam em
`appendix-A/01_main-chapter-code/DDP-script.py` no repositório oficial.